# 04b — ONNX Export & Benchmarking

Standalone notebook to:
1. Load a trained Transformer checkpoint
2. Re-export to ONNX
3. Validate parity (target: MAE diff < 1e-5)
4. Benchmark latency at different batch sizes
5. Profile memory footprint


In [ ]:
import sys
sys.path.insert(0, '..')
import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = 'plotly_white'

from backend.config import settings
from backend.ml.transformer_model import LTVTransformer, build_model
from backend.ml.trainer import LTVTrainer
from backend.ml.transformer_onnx import export_to_onnx, validate_onnx_vs_pytorch, ONNXInferenceEngine

ONNX_PATH  = settings.MODELS_DIR / 'transformer.onnx'
CKPT_PATH  = settings.MODELS_DIR / 'transformer_best.pt'
MAX_SEQ    = 50

print(f'ONNX path:  {ONNX_PATH}')
print(f'Checkpoint: {CKPT_PATH}')

In [ ]:
# Load checkpoint
ckpt = torch.load(CKPT_PATH, map_location='cpu')
config = ckpt['config']
model  = build_model(config)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded checkpoint — best val loss: {ckpt["best_val_loss"]:.6f}')

In [ ]:
# Export
onnx_path = export_to_onnx(model, ONNX_PATH, max_seq_len=MAX_SEQ)
print(f'Exported to {onnx_path}')

# Validate
val = validate_onnx_vs_pytorch(model, ONNX_PATH, max_seq_len=MAX_SEQ,
                                n_test_samples=500, tolerance=1e-4)
print(f'Parity: {"✓ PASSED" if val["passed"] else "✗ FAILED"}')
print(f'MAE diff: {val["mae_diff"]:.2e}')
print(f'Speedup: {val["speedup"]:.2f}×')

In [ ]:
# Latency at different batch sizes
import time
import onnxruntime as ort

session = ort.InferenceSession(str(ONNX_PATH))
batch_sizes = [1, 4, 8, 16, 32, 64, 128]
results = []

for bs in batch_sizes:
    dummy = {k: np.zeros((bs, MAX_SEQ), dtype=np.int64)
             for k in ['cat_id','amount_bucket','days_delta','channel_id']}
    # Warmup
    for _ in range(5): session.run(None, dummy)
    # Benchmark
    times = []
    for _ in range(100):
        t0 = time.perf_counter()
        session.run(None, dummy)
        times.append((time.perf_counter() - t0) * 1000)
    
    results.append({
        'batch_size': bs,
        'total_ms':   np.median(times),
        'per_sample_ms': np.median(times) / bs,
    })

import polars as pl
results_df = pl.DataFrame(results)
display(results_df)

fig = make_subplots(rows=1, cols=2, subplot_titles=['Total Latency', 'Per-Sample Latency'])
fig.add_trace(go.Bar(x=results_df['batch_size'].to_list(),
                      y=results_df['total_ms'].to_list(), name='Total'), row=1, col=1)
fig.add_trace(go.Bar(x=results_df['batch_size'].to_list(),
                      y=results_df['per_sample_ms'].to_list(), name='Per sample'), row=1, col=2)
fig.add_hline(y=200, row=1, col=1, line_dash='dash', line_color='red', annotation_text='200ms target')
fig.update_layout(height=380, title='ONNX Runtime Latency vs Batch Size', showlegend=False)
fig.show()